# PathLens-GNN — one method per run

Set `METHOD` to a directory name under `methods/`. The committed default is `three_hop` with `STAGE=eval` (full validation suite). `STAGE=final` is refused. Do not open the sealed test.

The branch must already be on GitHub (`GIT_REF`). Enable Internet. Accelerator: GPU T4. Run one method per session. After eval, the last cell writes PNG figures under `/kaggle/working/figures`.

In [ ]:
METHOD = "three_hop"  # folder name under methods/
STAGE = "eval"  # smoke | train | eval | final — eval completes the heuristic card
GIT_REF = "research/ranking-loss"
RESUME_ARCHIVE = None
FINAL_TEST_TOKEN = ""
DEVICE = "cuda:0"


In [ ]:
import os
import pathlib
import subprocess
import sys

REPO = pathlib.Path("/kaggle/working/PathLens-GNN")
if not REPO.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--filter=blob:none",
            "https://github.com/aryonmt/PathLens-GNN.git",
            str(REPO),
        ],
        check=True,
    )
subprocess.run(["git", "-C", str(REPO), "fetch", "origin", GIT_REF], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", "FETCH_HEAD"], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps"], check=True)
print(f"METHOD={METHOD} STAGE={STAGE} DEVICE={DEVICE}")


In [ ]:
from pathlens.runtime.runner import run_stage

result = run_stage(
    METHOD,
    STAGE,
    device=DEVICE,
    final_test_token=FINAL_TEST_TOKEN,
)
ranking = result["filtered_ranking"]
hard = result["classification"]["hard"]
print(result["output_dir"])
print(result.get("archive"))
print(
    f"device={result['device']} mrr={ranking['mrr']:.4f} "
    f"hits@10={ranking['hits_at_10']:.4f} ndcg@10={ranking['ndcg_at_10']:.4f} "
    f"hard_auprc={hard['auprc']:.4f}"
)
print(ranking)

In [ ]:
from pathlib import Path

from pathlens.evaluation.figures import load_validation_report, write_validation_figures

report = load_validation_report(Path("runs/biosnap-dti-v2"))
output = Path("/kaggle/working/figures")
written = write_validation_figures(report, output)
print("models:", ", ".join(sorted(report["models"])))
for path in written:
    print(path)